# Inicio

In [ ]:
pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.5 MB/s eta 0:00:00


In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.optim as optim
from torchmetrics.functional.classification import multiclass_f1_score
from sklearn.metrics import f1_score

import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import copy
from copy import deepcopy
from tqdm import tqdm
import time
import os
from scipy.signal import butter, sosfiltfilt
from scipy.spatial import distance
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [ ]:
def recortar_janelas(acc, J, passo):
    N = acc.shape[0]
    Nj = (N - J) // passo + 1
    janelas = np.zeros((Nj, J, 3))
    for i in range(Nj):
        janelas[i] = acc[i * passo:i * passo + J]
    return janelas

In [ ]:
actis = ['climbingdown', 'climbingup', 'jumping', 'lying', 'running', 'sitting', 'standing', 'walking']
posis = ['chest', 'forearm', 'head', 'shin', 'thigh', 'upperarm', 'waist']
users = ['proband' + x for x in np.arange(1,16).astype(str)]

In [ ]:
data = np.load('/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/notebooks/Xydata.npz')
Xdata = data['Xdata']
ydata = data['ydata']

In [ ]:
# pasta = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/Dataset/realworldcsvs/'
# J = 150
# Xdata = []
# ydata = []
# for user in tqdm(users):
#     for i, pos in enumerate(posis):
#         for j, act in enumerate(actis):
#             files = os.listdir(pasta+user+'/acc/')
#             inds = [(file.find(act)>-1) and (file.find(pos)>-1) for file in files]
#             if np.array(inds).any():
#                 ind = inds.index(True)
#                 acc = pd.read_csv(pasta+user+'/acc/'+files[ind]).values[:,2:]
#                 janelas = recortar_janelas(acc, J, J)
#                 rotulos = np.full((janelas.shape[0], 2), [i, j], dtype=int)
#                 Xdata.append(janelas)
#                 ydata.append(rotulos)
# Xdata = np.concatenate(Xdata, axis=0)
# Xdata = Xdata/20
# ydata = np.concatenate(ydata, axis=0)
# Xdata.shape, ydata.shape

# Modelo chang

In [ ]:
class ChangEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(in_channels=3, out_channels=16, kernel_size=3)
        self.inst1 = nn.InstanceNorm1d(16, affine=True)
        self.drop1 = nn.Dropout(p=0.2)

        self.conv2 = nn.Conv1d(in_channels=16, out_channels=16, kernel_size=3)
        self.inst2 = nn.InstanceNorm1d(16, affine=True)
        self.drop2 = nn.Dropout(p=0.2)

        self.conv3 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, stride=4)
        self.inst3 = nn.InstanceNorm1d(32, affine=True)
        self.drop3 = nn.Dropout(p=0.2)

        self.conv4 = nn.Conv1d(in_channels=32, out_channels=32, kernel_size=3, stride=1)
        self.inst4 = nn.InstanceNorm1d(32, affine=True)
        self.drop4 = nn.Dropout(p=0.2)

        self.conv5 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, stride=4)
        self.inst5 = nn.InstanceNorm1d(64, affine=True)
        self.drop5 = nn.Dropout(p=0.2)

        self.conv6 = nn.Conv1d(in_channels=64, out_channels=100, kernel_size=5, stride=1)

        self.relu = nn.LeakyReLU(0.3)
        self.glap = nn.AvgPool1d(kernel_size=4)

    def forward(self, x):
        # (N,T,C) -> (N,C,T)
        x = x.transpose(1, 2)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.inst1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.inst2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.relu(x)
        x = self.inst3(x)
        x = self.drop3(x)

        x = self.conv4(x)
        x = self.relu(x)
        x = self.inst4(x)
        x = self.drop4(x)

        x = self.conv5(x)
        x = self.relu(x)
        x = self.inst5(x)
        x = self.drop5(x)

        x = self.conv6(x)
        x = self.relu(x)

        x = self.glap(x)
        x = x.flatten(start_dim=1)

        logits = x
        return logits

In [ ]:
class ChangClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.densa = nn.Linear(in_features=100, out_features=8)

    def forward(self, x):
        logits = self.densa(x)
        return logits

# Funções de Treinamento

In [ ]:
inds = ydata[:,0]==0
X = Xdata[inds]
y = ydata[inds][:,1]
X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=1, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.1, random_state=1, stratify=y_train)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=125, shuffle=True, pin_memory=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False, pin_memory=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)

In [ ]:
@torch.no_grad()
def evaluate(encoder, classifier, loader, loss_fn, device):
    encoder.eval()
    classifier.eval()
    total_loss = 0
    total_samples = 0
    y_true = []
    y_pred = []

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = classifier(encoder(X))
        loss = loss_fn(logits, y)
        total_loss += loss.item() * len(y)
        total_samples += len(y)
        pred = torch.argmax(logits, dim=1)
        y_true.append(y)
        y_pred.append(pred)

    y_true = torch.cat(y_true)
    y_pred = torch.cat(y_pred)
    f1 = multiclass_f1_score(y_pred, y_true, num_classes=8, average="macro").item()

    return total_loss / total_samples, f1

In [ ]:
def train_model(train_loader, val_loader, device):
    encoder = ChangEncoder().to(device)
    classifier = ChangClassifier().to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(list(encoder.parameters()) + list(classifier.parameters()), lr=1e-3)
    n_epochs = 100
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_f1": [],
        "val_f1": []
    }
    best_f1 = -1
    best_encoder = None
    best_classifier = None

    for epoch in range(n_epochs):
        encoder.train()
        classifier.train()
        running_loss = 0
        n_samples = 0
        bar = tqdm(train_loader)

        for X, y in bar:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad()
            logits = classifier(encoder(X))
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(y)
            n_samples += len(y)
            bar.set_description(f"Epoch {epoch+1}")
            bar.set_postfix(loss=loss.item())

        train_loss, train_f1 = evaluate(
            encoder,
            classifier,
            train_loader,
            loss_fn,
            device
        )

        val_loss, val_f1 = evaluate(
            encoder,
            classifier,
            val_loader,
            loss_fn,
            device
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)

        print(
            f"Epoch {epoch+1:3d} | "
            f"Train F1={train_f1:.4f} | "
            f"Val F1={val_f1:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_encoder = deepcopy(encoder)
            best_classifier = deepcopy(classifier)

    return best_encoder, best_classifier, history

# Teste baseline

In [ ]:
# Resultados de referência
data = {
    'Treino': [86, 94, 89, 88, 93, 94, 92],
    'head': ['-', 28, 47, 16, 16, 39, 6],
    'chest': [60, '-', 58, 12, 22, 39, 29],
    'upperarm': [51, 32, '-', 11, 5, 45, 33],
    'forearm': [26, 25, 20, '-', 25, 5, 3],
    'waist': [14, 30, 18, 39, '-', 16, 9],
    'thigh': [37, 42, 44, 12, 15, '-', 30],
    'shin': [23, 24, 35, 9, 2, 40, '-']
}

index_labels = ['head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin']

df = pd.DataFrame(data, index=index_labels)

display(df)

## Baseline 0: Chang

In [ ]:
pasta = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/'
baselinef1 = np.zeros((7,7))
for i, pos in enumerate(posis):
    inds = ydata[:,0]==i
    X = Xdata[inds]
    y = ydata[inds][:,1]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test  = torch.tensor(X_test, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test  = torch.tensor(y_test, dtype=torch.long)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=125, shuffle=True, pin_memory=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
    enc, cla, history = train_model(train_loader, test_loader, device)
    nome = 'baseline_encoder_'+pos+'.pth'
    torch.save(enc.state_dict(), pasta+nome)
    nome = 'baseline_classifier_'+pos+'.pth'
    torch.save(cla.state_dict(), pasta+nome)
    for j in range(7):
        inds = ydata[:,0]==j
        X = Xdata[inds]
        y = ydata[inds][:,1]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
        X_test  = torch.tensor(X_test, dtype=torch.float32)
        y_test  = torch.tensor(y_test, dtype=torch.long)
        test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
        _, test_f1 = evaluate(enc, cla, test_loader, nn.CrossEntropyLoss(), device)
        baselinef1[i,j] = test_f1

Epoch 1: 100%|██████████| 139/139 [00:01<00:00, 96.92it/s, loss=1.07] 


Epoch   1 | Train F1=0.5213 | Val F1=0.5164


Epoch 2: 100%|██████████| 139/139 [00:01<00:00, 85.47it/s, loss=0.751]


Epoch   2 | Train F1=0.6075 | Val F1=0.5980


Epoch 3: 100%|██████████| 139/139 [00:01<00:00, 87.74it/s, loss=0.653]


...

Epoch  98 | Train F1=0.9534 | Val F1=0.9071


Epoch 99: 100%|██████████| 139/139 [00:01<00:00, 80.31it/s, loss=0.331]


Epoch  99 | Train F1=0.9493 | Val F1=0.9067


Epoch 100: 100%|██████████| 139/139 [00:01<00:00, 78.94it/s, loss=0.176]


Epoch 100 | Train F1=0.9566 | Val F1=0.9091


In [ ]:
df = pd.DataFrame((baselinef1*100).astype(int), columns=posis, index=posis)
df.insert(0, 'Treino', (baselinef1.diagonal()*100).astype(int))
for i in range(7):
    df.iloc[i,i+1] = '-'
df = df.reindex(index=['head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin'])
df = df.reindex(columns=['Treino','head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin'])
df

In [ ]:
data = {
    'Treino': [84, 90, 88, 84, 91, 90, 91],
    'head': ['-', 46, 46, 30, 38, 41, 18],
    'chest': [52, '-', 64, 32, 29, 34, 34],
    'upperarm': [51, 43, '-', 39, 20, 39, 39],
    'forearm': [25, 38, 29, '-', 27, 33, 28],
    'waist': [29, 37, 18, 24, '-', 39, 21],
    'thigh': [44, 46, 62, 35, 28, '-', 38],
    'shin': [22, 35, 46, 28, 17, 51, '-']
}

index_labels = ['head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin']

df = pd.DataFrame(data, index=index_labels)

display(df)

,Treino,head,chest,upperarm,forearm,waist,thigh,shin
head,84,-,52,51,25,29,44,22
chest,90,46,-,43,38,37,46,35
upperarm,88,46,64,-,29,18,62,46
forearm,84,30,32,39,-,24,35,28
waist,91,38,29,20,27,-,28,17
thigh,90,41,34,39,33,39,-,51
shin,91,18,34,39,28,21,38,-


## Baseline 1: filtro passa-altas

In [ ]:
sos = butter(N=6, Wn=0.3, btype='hp', fs=50, output='sos')
Xf = np.swapaxes(sosfiltfilt(sos, np.swapaxes(Xdata, 1, 2)), 1, 2)

In [ ]:
pasta = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/'
baselinef1 = np.zeros((7,7))
for i, pos in enumerate(posis):
    inds = ydata[:,0]==i
    X = Xf[inds]
    y = ydata[inds][:,1]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test  = torch.tensor(X_test, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test  = torch.tensor(y_test, dtype=torch.long)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=125, shuffle=True, pin_memory=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
    enc, cla, history = train_model(train_loader, test_loader, device)
    nome = 'baseline_hp_encoder_'+pos+'.pth'
    torch.save(enc.state_dict(), pasta+nome)
    nome = 'baseline_hp_classifier_'+pos+'.pth'
    torch.save(cla.state_dict(), pasta+nome)
    for j in range(7):
        inds = ydata[:,0]==j
        X = Xf[inds]
        y = ydata[inds][:,1]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
        X_test  = torch.tensor(X_test, dtype=torch.float32)
        y_test  = torch.tensor(y_test, dtype=torch.long)
        test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
        _, test_f1 = evaluate(enc, cla, test_loader, nn.CrossEntropyLoss(), device)
        baselinef1[i,j] = test_f1

In [ ]:
df = pd.DataFrame((baselinef1*100).astype(int), columns=posis, index=posis)
df.insert(0, 'Treino', (baselinef1.diagonal()*100).astype(int))
for i in range(7):
    df.iloc[i,i+1] = '-'
df = df.reindex(index=['head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin'])
df = df.reindex(columns=['Treino','head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin'])

In [ ]:
data = {
    'Treino': [84, 90, 88, 84, 91, 90, 91],
    'head': ['-', 46, 46, 30, 38, 41, 18],
    'chest': [52, '-', 64, 32, 29, 34, 34],
    'upperarm': [51, 43, '-', 39, 20, 39, 39],
    'forearm': [25, 38, 29, '-', 27, 33, 28],
    'waist': [29, 37, 18, 24, '-', 39, 21],
    'thigh': [44, 46, 62, 35, 28, '-', 38],
    'shin': [22, 35, 46, 28, 17, 51, '-']
}

index_labels = ['head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin']

df0 = pd.DataFrame(data, index=index_labels)

print(df)
print(df0)

          Treino head chest upperarm forearm waist thigh shin
head          82    -    44       46      32    34    38   20
chest         84   51     -       52      39    32    48   37
upperarm      85   46    61        -      32    17    56   43
forearm       81   41    43       46       -    31    37   39
waist         86   43    28       21      25     -    27   18
thigh         84   43    37       43      36    35     -   49
shin          87   22    26       38      28    15    29    -
          Treino head chest upperarm forearm waist thigh shin
head          84    -    52       51      25    29    44   22
chest         90   46     -       43      38    37    46   35
upperarm      88   46    64        -      29    18    62   46
forearm       84   30    32       39       -    24    35   28
waist         91   38    29       20      27     -    28   17
thigh         90   41    34       39      33    39     -   51
shin          91   18    34       39      28    21    38    -


## Baseline 2: filtro passa-baixas 10 Hz

In [ ]:
sos = butter(N=6, Wn=10, btype='lp', fs=50, output='sos')
Xf = np.swapaxes(sosfiltfilt(sos, np.swapaxes(Xdata, 1, 2)), 1, 2)

In [ ]:
pasta = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/'
baselinef1 = np.zeros((7,7))
for i, pos in enumerate(posis):
    inds = ydata[:,0]==i
    X = Xf[inds]
    y = ydata[inds][:,1]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test  = torch.tensor(X_test, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test  = torch.tensor(y_test, dtype=torch.long)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=125, shuffle=True, pin_memory=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
    enc, cla, history = train_model(train_loader, test_loader, device)
    nome = 'baseline_lp_encoder_'+pos+'.pth'
    torch.save(enc.state_dict(), pasta+nome)
    nome = 'baseline_lp_classifier_'+pos+'.pth'
    torch.save(cla.state_dict(), pasta+nome)
    for j in range(7):
        inds = ydata[:,0]==j
        X = Xf[inds]
        y = ydata[inds][:,1]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
        X_test  = torch.tensor(X_test, dtype=torch.float32)
        y_test  = torch.tensor(y_test, dtype=torch.long)
        test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
        _, test_f1 = evaluate(enc, cla, test_loader, nn.CrossEntropyLoss(), device)
        baselinef1[i,j] = test_f1

Epoch 1: 100%|██████████| 139/139 [00:02<00:00, 48.01it/s, loss=1.02]


Epoch   1 | Train F1=0.5223 | Val F1=0.5213


Epoch 2: 100%|██████████| 139/139 [00:03<00:00, 46.23it/s, loss=0.882]


Epoch   2 | Train F1=0.6303 | Val F1=0.6173


Epoch 3: 100%|██████████| 139/139 [00:02<00:00, 61.29it/s, loss=0.649]


...

Epoch  98 | Train F1=0.9538 | Val F1=0.9089


Epoch 99: 100%|██████████| 139/139 [00:01<00:00, 91.84it/s, loss=0.156]


Epoch  99 | Train F1=0.9558 | Val F1=0.9130


Epoch 100: 100%|██████████| 139/139 [00:01<00:00, 83.60it/s, loss=0.189]


Epoch 100 | Train F1=0.9546 | Val F1=0.9109


In [ ]:
df = pd.DataFrame((baselinef1*100).astype(int), columns=posis, index=posis)
df.insert(0, 'Treino', (baselinef1.diagonal()*100).astype(int))
for i in range(7):
    df.iloc[i,i+1] = '-'
df = df.reindex(index=['head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin'])
df = df.reindex(columns=['Treino','head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin'])

In [ ]:
data = {
    'Treino': [84, 90, 88, 84, 91, 90, 91],
    'head': ['-', 46, 46, 30, 38, 41, 18],
    'chest': [52, '-', 64, 32, 29, 34, 34],
    'upperarm': [51, 43, '-', 39, 20, 39, 39],
    'forearm': [25, 38, 29, '-', 27, 33, 28],
    'waist': [29, 37, 18, 24, '-', 39, 21],
    'thigh': [44, 46, 62, 35, 28, '-', 38],
    'shin': [22, 35, 46, 28, 17, 51, '-']
}

index_labels = ['head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin']

df0 = pd.DataFrame(data, index=index_labels)

print(df)
print(df0)

          Treino head chest upperarm forearm waist thigh shin
head          83    -    47       53      32    35    39   31
chest         89   42     -       44      35    34    46   39
upperarm      87   47    62        -      28    24    58   52
forearm       85   33    39       36       -    20    34   35
waist         91   37    25       18      23     -    22   16
thigh         91   41    30       38      31    42     -   48
shin          91   19    33       41      28    15    41    -
          Treino head chest upperarm forearm waist thigh shin
head          84    -    52       51      25    29    44   22
chest         90   46     -       43      38    37    46   35
upperarm      88   46    64        -      29    18    62   46
forearm       84   30    32       39       -    24    35   28
waist         91   38    29       20      27     -    28   17
thigh         90   41    34       39      33    39     -   51
shin          91   18    34       39      28    21    38    -


## Baseline 3: passa-baixas 2 Hz

In [ ]:
sos = butter(N=6, Wn=2, btype='lp', fs=50, output='sos')
Xf = np.swapaxes(sosfiltfilt(sos, np.swapaxes(Xdata, 1, 2)), 1, 2)

In [ ]:
pasta = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/'
baselinef1 = np.zeros((7,7))
for i, pos in enumerate(posis):
    inds = ydata[:,0]==i
    X = Xf[inds]
    y = ydata[inds][:,1]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test  = torch.tensor(X_test, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test  = torch.tensor(y_test, dtype=torch.long)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=125, shuffle=True, pin_memory=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
    enc, cla, history = train_model(train_loader, test_loader, device)
    nome = 'baseline_lp2Hz_encoder_'+pos+'.pth'
    torch.save(enc.state_dict(), pasta+nome)
    nome = 'baseline_lp2Hz_classifier_'+pos+'.pth'
    torch.save(cla.state_dict(), pasta+nome)
    for j in range(7):
        inds = ydata[:,0]==j
        X = Xf[inds]
        y = ydata[inds][:,1]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
        X_test  = torch.tensor(X_test, dtype=torch.float32)
        y_test  = torch.tensor(y_test, dtype=torch.long)
        test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False, pin_memory=True)
        _, test_f1 = evaluate(enc, cla, test_loader, nn.CrossEntropyLoss(), device)
        baselinef1[i,j] = test_f1

Epoch 1: 100%|██████████| 139/139 [00:04<00:00, 29.50it/s, loss=1.06]


Epoch   1 | Train F1=0.5308 | Val F1=0.5296


Epoch 2: 100%|██████████| 139/139 [00:01<00:00, 72.45it/s, loss=0.794]


Epoch   2 | Train F1=0.6205 | Val F1=0.6207


Epoch 3: 100%|██████████| 139/139 [00:01<00:00, 80.23it/s, loss=0.751]


...

Epoch  98 | Train F1=0.9402 | Val F1=0.8963


Epoch 99: 100%|██████████| 139/139 [00:01<00:00, 76.30it/s, loss=0.267]


Epoch  99 | Train F1=0.9408 | Val F1=0.8977


Epoch 100: 100%|██████████| 139/139 [00:01<00:00, 83.66it/s, loss=0.31]


Epoch 100 | Train F1=0.9387 | Val F1=0.8975


In [ ]:
df = pd.DataFrame((baselinef1*100).astype(int), columns=posis, index=posis)
df.insert(0, 'Treino', (baselinef1.diagonal()*100).astype(int))
for i in range(7):
    df.iloc[i,i+1] = '-'
df = df.reindex(index=['head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin'])
df = df.reindex(columns=['Treino','head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin'])

In [ ]:
data = {
    'Treino': [84, 90, 88, 84, 91, 90, 91],
    'head': ['-', 46, 46, 30, 38, 41, 18],
    'chest': [52, '-', 64, 32, 29, 34, 34],
    'upperarm': [51, 43, '-', 39, 20, 39, 39],
    'forearm': [25, 38, 29, '-', 27, 33, 28],
    'waist': [29, 37, 18, 24, '-', 39, 21],
    'thigh': [44, 46, 62, 35, 28, '-', 38],
    'shin': [22, 35, 46, 28, 17, 51, '-']
}

index_labels = ['head', 'chest', 'upperarm', 'forearm', 'waist', 'thigh', 'shin']

df0 = pd.DataFrame(data, index=index_labels)

print(df)
print(df0)

          Treino head chest upperarm forearm waist thigh shin
head          80    -    43       49      29    43    46   28
chest         87   36     -       48      37    31    48   36
upperarm      85   43    59        -      30    19    44   48
forearm       82   29    42       44       -    21    36   36
waist         90   31    23       16      32     -    31   18
thigh         87   39    36       39      30    33     -   39
shin          90   23    47       51      32    25    41    -
          Treino head chest upperarm forearm waist thigh shin
head          84    -    52       51      25    29    44   22
chest         90   46     -       43      38    37    46   35
upperarm      88   46    64        -      29    18    62   46
forearm       84   30    32       39       -    24    35   28
waist         91   38    29       20      27     -    28   17
thigh         90   41    34       39      33    39     -   51
shin          91   18    34       39      28    21    38    -


# Treinamento de adaptação

In [ ]:
vals = [0.01, 0.1, 1, 2, 3, 4, 5, 6, 10, 20, 50, 100, 200, 250, 300, 400, 500, 1000]
epochs = 100
batch_size = 125

In [ ]:
def mmd2u(x, y, c):
    n = x.shape[0]
    m = y.shape[0]
    xy = torch.vstack((x,y))
    dists = torch.cdist(xy, xy)
    k = torch.exp( (-1/(2*c)) * dists**2 )
    k_x = torch.triu(k[:n, :n], diagonal=1)
    k_y = torch.triu(k[n:, n:], diagonal=1)
    k_xy = k[:n, n:]
    mmd = 2*k_x.sum()/(n*(n-1)) + 2*k_y.sum()/(m*(m-1)) - 2*k_xy.sum()/(n*m)
    return mmd

In [ ]:
def train_feature_matching(Xs_train, ys_train, Xt_train, Xs_val, ys_val, encoder, classifier):
    if encoder is None:
        encoder = ChangEncoder().to(device)
    if classifier is None:
        classifier = ChangClassifier().to(device)
    loss_fn = nn.CrossEntropyLoss()
    opt_cls = torch.optim.Adam(list(encoder.parameters()) + list(classifier.parameters()), lr=1e-3)
    opt_mmd = torch.optim.Adam(encoder.parameters(), lr=1e-3)
    history = {"train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}
    Ns = len(Xs_train)
    Nt = len(Xt_train)
    for epoch in range(epochs):
        encoder.train()
        classifier.train()
        perm_s = torch.randperm(Ns, device=device)
        perm_t = torch.randperm(Nt, device=device)
        Xs = Xs_train[perm_s]
        ys = ys_train[perm_s]
        Xt = Xt_train[perm_t]
        n_batches = min(Ns, Nt) // batch_size
        pbar = tqdm(range(n_batches), desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        for b in pbar:
            i0 = b * batch_size
            i1 = i0 + batch_size
            Xsb = Xs[i0:i1]
            ysb = ys[i0:i1]
            Xtb = Xt[i0:i1]
            # PASSO 1: classificação
            opt_cls.zero_grad()
            feat = encoder(Xsb)
            logits = classifier(feat)
            loss_cls = loss_fn(logits, ysb)
            loss_cls.backward()
            opt_cls.step()
            # PASSO 2: feature matching
            opt_mmd.zero_grad()
            feat_s = encoder(Xsb)
            feat_t = encoder(Xtb)
            loss_mmd = 0.0
            for sigma2 in vals:
                loss_mmd += mmd2u(feat_s, feat_t, sigma2)
            loss_mmd.backward()
            opt_mmd.step()
            pbar.set_postfix(cls=f"{loss_cls.item():.4f}", mmd=f"{loss_mmd.item():.4f}")
        encoder.eval()
        classifier.eval()
        with torch.no_grad():
            logits = classifier(encoder(Xs_train))
            train_loss = loss_fn(logits, ys_train)
            train_pred = logits.argmax(1)
            train_f1 = multiclass_f1_score(train_pred, ys_train, num_classes=8)
            logits = classifier(encoder(Xs_val))
            val_loss = loss_fn(logits, ys_val)
            val_pred = logits.argmax(1)
            val_f1 = multiclass_f1_score(val_pred, ys_val, num_classes=8)
        history["train_loss"].append(train_loss.item())
        history["train_f1"].append(train_f1.item())
        history["val_loss"].append(val_loss.item())
        history["val_f1"].append(val_f1.item())
        print(
            f"Epoch {epoch+1:2d} | "
            f"train F1={train_f1:.4f} | "
            f"val F1={val_f1:.4f}"
        )
    return encoder, classifier, history

In [ ]:
s = 0
t = 1
inds = ydata[:,0]==s
Xs = Xdata[inds]
ys = ydata[inds][:,1]
Xs_train, Xs_test, ys_train, ys_test = train_test_split(Xs, ys, test_size=0.2, random_state=1, stratify=ys)
Xs_train = torch.tensor(Xs_train, dtype=torch.float32).to(device)
Xs_test  = torch.tensor(Xs_test, dtype=torch.float32).to(device)
ys_train = torch.tensor(ys_train, dtype=torch.long).to(device)
ys_test  = torch.tensor(ys_test, dtype=torch.long).to(device)
inds = ydata[:,0]==t
Xt = Xdata[inds]
yt = ydata[inds][:,1]
Xt_train, Xt_test, yt_train, yt_test = train_test_split(Xt, yt, test_size=0.2, random_state=1, stratify=yt)
Xt_train = torch.tensor(Xt_train, dtype=torch.float32).to(device)
Xt_test  = torch.tensor(Xt_test, dtype=torch.float32).to(device)
yt_train = torch.tensor(yt_train, dtype=torch.long).to(device)
yt_test  = torch.tensor(yt_test, dtype=torch.long).to(device)

In [ ]:
enc, cla, history = train_feature_matching(Xs_train, ys_train, Xt_train, Xs_test, ys_test, None, None)

Epoch  1 | train F1=0.5202 | val F1=0.5156


Epoch  2 | train F1=0.6071 | val F1=0.6128


Epoch  3 | train F1=0.6780 | val F1=0.6600


Epoch  4 | train F1=0.6655 | val F1=0.6523


Epoch  5 | train F1=0.7109 | val F1=0.6821


...

Epoch 96 | train F1=0.9288 | val F1=0.8877


Epoch 97 | train F1=0.9269 | val F1=0.8839


Epoch 98 | train F1=0.9292 | val F1=0.8909


Epoch 99 | train F1=0.9299 | val F1=0.8868


Epoch 100 | train F1=0.9317 | val F1=0.8931


In [ ]:
source_loader = DataLoader(TensorDataset(Xs_test, ys_test), batch_size=125)
target_loader = DataLoader(TensorDataset(Xt_test, yt_test), batch_size=125)
_, f1s = evaluate(enc, cla, source_loader, nn.CrossEntropyLoss(), device)
_, f1t = evaluate(enc, cla, target_loader, nn.CrossEntropyLoss(), device)
doms = posis[s]
domt = posis[t]
print('Antes da adaptação \tDepois da adaptação')
print('F1 '+doms+':', df['Treino'][doms]/100, '  \tF1 '+doms+':', np.array(f1s).round(2))
print('F1 '+domt+':', df[domt][doms]/100, '\tF1 '+doms+':', np.array(f1t).round(2))

Antes da adaptação 	Depois da adaptação
F1 chest: 0.9   	F1 chest: 0.89
F1 forearm: 0.38 	F1 chest: 0.52


In [ ]:
pasta = '/content/drive/MyDrive/Doutorado Unicamp/Projeto/github/chang-UDA-HAR/modelos/'
enc2 = ChangEncoder().to(device)
enc2.load_state_dict(torch.load(pasta+'baseline_encoder_chest.pth'))
cla2 = ChangClassifier().to(device)
cla2.load_state_dict(torch.load(pasta+'baseline_classifier_chest.pth'))

<All keys matched successfully>

In [ ]:
enc2, cla2, history = train_feature_matching(Xs_train, ys_train, Xt_train, Xs_test, ys_test, enc2, cla2)

Epoch  1 | train F1=0.9338 | val F1=0.8792


Epoch  2 | train F1=0.9382 | val F1=0.8934


Epoch  3 | train F1=0.9388 | val F1=0.9019


Epoch  4 | train F1=0.9374 | val F1=0.8919


Epoch  5 | train F1=0.9375 | val F1=0.8992


...

Epoch 96 | train F1=0.9533 | val F1=0.9045


Epoch 97 | train F1=0.9562 | val F1=0.9080


Epoch 98 | train F1=0.9480 | val F1=0.8936


Epoch 99 | train F1=0.9572 | val F1=0.9049


Epoch 100 | train F1=0.9543 | val F1=0.9009


In [ ]:
source_loader = DataLoader(TensorDataset(Xs_test, ys_test), batch_size=125)
target_loader = DataLoader(TensorDataset(Xt_test, yt_test), batch_size=125)
_, f1s = evaluate(enc2, cla2, source_loader, nn.CrossEntropyLoss(), device)
_, f1t = evaluate(enc2, cla2, target_loader, nn.CrossEntropyLoss(), device)
doms = posis[s]
domt = posis[t]
print('Antes da adaptação \tDepois da adaptação')
print('F1 '+doms+':', df['Treino'][doms]/100, '  \tF1 '+doms+':', np.array(f1s).round(2))
print('F1 '+domt+':', df[domt][doms]/100, '\tF1 '+doms+':', np.array(f1t).round(2))

Antes da adaptação 	Depois da adaptação
F1 chest: 0.9   	F1 chest: 0.9
F1 forearm: 0.38 	F1 chest: 0.45
